# Feature Engineering: Scaling, Extraction, and Selection
**Comprehensive Master Notebook** *Last Updated: April 2026*

This notebook provides an end-to-end guide to preparing raw datasets for Machine Learning models. We will explore how to change data scale, extract hidden structural components, and select the most impactful features to prevent overfitting and accelerate training times.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.preprocessing import MinMaxScaler, Normalizer, StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Set random seed for reproducibility
np.random.seed(42)

# Generate a base classification dataset (1000 samples, 6 raw variables)
X_raw, y = make_classification(
    n_samples=1000, 
    n_features=6, 
    n_informative=4, 
    n_redundant=2, 
    random_state=42
)

# Convert to DataFrame and manually inject diverse scales and outliers
df = pd.DataFrame(X_raw, columns=['feat_1', 'feat_2', 'feat_3', 'feat_4', 'feat_5', 'feat_6'])

# Inject real-world variations:
df['feat_1'] = df['feat_1'] * 10000          # Large magnitude scale
df['feat_2'] = df['feat_2'] + 50             # Shifted mean
df['feat_3'] = np.exp(df['feat_3'])          # Highly skewed log-normal distribution
# Inject heavy extreme outliers into feature 4
df.loc[::50, 'feat_4'] = df.loc[::50, 'feat_4'] * 50 

print("--- Raw Unscaled Data Head ---")
print(df.head())

## 2. Feature Scaling & Normalization

Distance-based algorithms (like KNN, SVM) and gradient-descent optimization models (like Neural Networks, Logistic Regression) perform poorly or fail to converge if features exist on vastly different numerical spectrums.

### 2.1 Absolute Maximum Scaling
Rescales data dynamically between $[-1, 1]$. It divides every element by the maximum absolute value of that specific column.
$$\mathbf{X}_{\text{scaled}} = \frac{X_i}{\max(|X|)}$$
* **Best Used For:** Sparse data arrays (e.g., text processing bags-of-words).
* **Risk:** Highly vulnerable to damage from large unmitigated outliers.

In [ ]:
# Manual implementation of Absolute Maximum Scaling
max_abs = np.max(np.abs(df), axis=0)
df_abs_max = df / max_abs

print("Absolute Maximum Scaled Data Summary (Range ideally bounds within [-1, 1]):")
print(df_abs_max.describe().loc[['min', 'max']])

### 2.2 Min-Max Scaling (Normalization)
Transforms data to fit exactly within a predefined boundary—typically $[0, 1]$.
$$\mathbf{X}_{\text{scaled}} = \frac{X_i - X_{\min}}{X_{\max} - X_{\min}}$$
* **Best Used For:** Algorithms requiring bounded inputs (Neural Networks, Image Pixel values).

In [ ]:
min_max_scaler = MinMaxScaler()
df_min_max = pd.DataFrame(min_max_scaler.fit_transform(df), columns=df.columns)

print("\nMin-Max Scaled Data Summary (Strictly bounded [0, 1]):")
print(df_min_max.describe().loc[['min', 'max']])

### 2.3 Vector Normalizer
Scales individual rows (samples) independently rather than whole columns, ensuring that each row vector possesses a Euclidean Length ($L_2$ norm) equal to $1$.
$$\mathbf{X}_{\text{scaled}} = \frac{X_i}{\|X\|}$$
* **Best Used For:** Text classification similarity assessments or cosine-distance calculations.

In [ ]:
vector_normalizer = Normalizer(norm='l2')
df_vector_norm = pd.DataFrame(vector_normalizer.fit_transform(df), columns=df.columns)

# Verify that the Euclidean norm of row index 0 equals exactly 1.0
print(f"\nL2 Norm check for Sample 0: {np.linalg.norm(df_vector_norm.iloc[0]):.4f}")

### 2.4 Standardization (Z-Score Scaling)
Centers the underlying data to ensure a mean ($\mu$) of $0$ and a standard deviation ($\sigma$) of $1$.
$$\mathbf{X}_{\text{scaled}} = \frac{X_i - \mu}{\sigma}$$
* **Best Used For:** Algorithms assuming normal distributions (Linear Regression, Logistic Regression, LDA, PCA).

In [ ]:
std_scaler = StandardScaler()
df_standardized = pd.DataFrame(std_scaler.fit_transform(df), columns=df.columns)

print("\nStandardized Data Summary (Mean ≈ 0, Std Dev ≈ 1):")
print(df_standardized.describe().loc[['mean', 'std']].round(4))

### 2.5 Robust Scaling
Uses statistical quartiles (Median and Interquartile Range) to scale variables. 
$$\mathbf{X}_{\text{scaled}} = \frac{X_i - X_{\text{median}}}{\text{IQR}}$$
* **Best Used For:** Datasets contaminated with extreme, unpredictable real-world **outliers** (like `feat_4` in our dummy data).

In [ ]:
robust_scaler = RobustScaler()
df_robust = pd.DataFrame(robust_scaler.fit_transform(df), columns=df.columns)

print("\nRobust Scaled Data Summary (Unaffected by extreme outliers):")
print(df_robust.describe().loc[['50%', 'max']])

---
## 3. Feature Extraction via Dimensionality Reduction

Feature Extraction maps the original high-dimensional feature space down into a new, lower-dimensional space. We will use **Principal Component Analysis (PCA)** to capture the vast majority of the variance using orthogonal vectors.

In [ ]:
# Instantiate PCA to capture 90% of structural variance
pca = PCA(n_components=0.90, random_state=42)

# Important Note: Always standardize features BEFORE applying PCA
df_standardized_clean = df_standardized.fillna(0) # Safety check
pca_transformed = pca.fit_transform(df_standardized_clean)

# Structure results into a clean DataFrame
pca_cols = [f'PC_{i+1}' for i in range(pca_transformed.shape[1])]
df_pca = pd.DataFrame(pca_transformed, columns=pca_cols)

print(f"Original Feature Count: {df_standardized.shape[1]}")
print(f"Reduced PCA Feature Count (90% Variance Explained): {df_pca.shape[1]}")
print(f"Explained Variance Ratio per Component: {pca.explained_variance_ratio_}")

---
## 4. Feature Selection Methods

Unlike Extraction, Feature Selection evaluates existing variables and extracts the most valuable subset without transforming their meaning, preserving data interpretability.

### 4.1 Filter Method: ANOVA F-test Variance
Evaluates each feature independently of others based on statistical tests against the target variable $y$.

In [ ]:
# Select the top 3 scoring columns using ANOVA F-value
filter_selector = SelectKBest(score_func=f_classif, k=3)
X_filtered = filter_selector.fit_transform(df_standardized, y)

selected_indices = filter_selector.get_support(indices=True)
print("Filter Method Selected Features:")
print(df.columns[selected_indices].tolist())

### 4.2 Wrapper Method: Recursive Feature Elimination (RFE)
An iterative process that uses an external estimator model to evaluate feature groups, systematically dropping the least impactful features one by one.

In [ ]:
# Initialize an estimator (Logistic Regression)
estimator_lr = LogisticRegression(solver='liblinear')

# Run RFE to isolate the top 3 features
rfe_selector = RFE(estimator=estimator_lr, n_features_to_select=3, step=1)
X_rfe = rfe_selector.fit(df_standardized, y)

print("Wrapper Method (RFE) Selected Features:")
print(df.columns[rfe_selector.support_].tolist())

### 4.3 Embedded Method: Tree-Based Feature Importance
Embedded selectors discover which features contribute most to performance directly during model training.

In [ ]:
# Train a Random Forest classifier directly on standardized data
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(df_standardized, y)

# Construct Feature Importance mapping
importances = rf_model.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': df.columns,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("Embedded Method (Random Forest) Rankings:")
print(feature_importance_df)

---
## 5. Summary Cheat Sheet

| Scaling/Selection Type | Primary Use Case | Sensitivity to Outliers | Retains Original Meaning? |
| :--- | :--- | :--- | :--- |
| **Min-Max Scaler** | Neural Networks, Bounded range inputs | High | Yes |
| **StandardScaler** | Linear models, SVMs, PCA preprocessing | Moderate | Yes |
| **RobustScaler** | Dirty datasets with heavy skew/outliers | Low | Yes |
| **PCA (Extraction)**| Drastic dimensionality compression | High | No (Creates combinations) |
| **Filter Methods** | Rapid, high-volume initial feature sorting| Varies | Yes |
| **Wrapper (RFE)** | Optimizing predictive accuracy on specific models| High | Yes |
| **Embedded (Trees)**| Non-linear data relationships | Low | Yes |